# Feature time series with seizure overlay

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

from seizure_onset_detector.data import (
    load_info,
    load_recording,
    recording_start_seconds,
    window_signal,
)

from seizure_onset_detector.features import(
    extract_features_batch,
    feature_names
)

p = Path.cwd()
while not (p / "pyproject.toml").exists():
    if p.parent == p:
        raise RuntimeError("project root not found")
    p = p.parent
os.chdir(p)

In [ ]:
DATA_DIR = "datasets/swec-ethz"
CASES = [("03", "84h"), ("18", "93h")]

PRESENT = {
    "line_length_max":     "Line length",
    "bp_gamma_max":        "Gamma power",
    "energy_ratio_max":    "Energy ratio",
    "sef95_max":           r"SEF$_{95}$ (Hz)",
    "hjorth_mobility_max": "Hjorth mobility",
}
to_plot = list(PRESENT.keys())
n_rows = 1 + len(to_plot)  # 1 raw + 5 features
n_cols = len(CASES)

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(7 * n_cols, 1.8 * n_rows),
                         sharex="col", sharey="row",
                         constrained_layout=True)

labelled = False
for col, (patient, RECORDING) in enumerate(CASES):
    fs, seizure_intervals = load_info(DATA_DIR, patient)
    signal = load_recording(DATA_DIR, patient, RECORDING)
    offset = recording_start_seconds(RECORDING)
    windows, start_times = window_signal(signal, fs, start_offset_sec=offset)
    features = extract_features_batch(windows, fs)
    names = feature_names()

    # Top row: raw iEEG (channel 1)
    sz_start, sz_end = seizure_intervals[0]
    sz_mask = (start_times >= sz_start) & (start_times <= sz_end)
    ll_per_channel = np.sum(np.abs(np.diff(windows[sz_mask], axis=2)), axis=2)  # (n_sz_windows, n_channels)
    best_channel = ll_per_channel.sum(axis=0).argmax()
    raw_time = np.arange(signal.shape[1]) / fs + offset
    axes[0, col].plot(raw_time, signal[best_channel], linewidth=0.2, color="steelblue")
    axes[0, col].set_title(f"Patient {patient}, {RECORDING}")

    # Feature rows
    for row, feat_name in enumerate(to_plot, start=1):
        idx = names.index(feat_name)
        axes[row, col].plot(start_times, features[:, idx], linewidth=0.5)
        axes[row, col].set_yscale("log")

    # Seizure overlay (only for seizures that fall in the visible time range)
    t0, t1 = start_times[0], start_times[-1]
    visible = [(s, e) for s, e in seizure_intervals if e > t0 and s < t1]
    for ax in axes[:, col]:
        for sz_start, sz_end in visible:
            ax.axvspan(sz_start, sz_end, alpha=0.3, color="crimson",
                       label="Seizure" if not labelled else None)
            labelled = True

    # Zoom column x-axis to ±60s around the first visible seizure
    if visible:
        sz_start, sz_end = visible[0]
        axes[0, col].set_xlim(sz_start - 60, sz_end + 60)

# Equalize column x-axis spans to the widest, recentered on each column's midpoint
xlims = [axes[0, c].get_xlim() for c in range(n_cols)]
max_span = max(hi - lo for lo, hi in xlims)
for c, (lo, hi) in enumerate(xlims):
    mid = (lo + hi) / 2
    axes[0, c].set_xlim(mid - max_span/2, mid + max_span/2)

# Y-labels on leftmost column only (sharey='row')
axes[0, 0].set_ylabel("iEEG\n(peak ch.)")
for row, feat_name in enumerate(to_plot, start=1):
    axes[row, 0].set_ylabel(PRESENT[feat_name])

# X-labels on bottom row only (sharex='col')
for col in range(n_cols):
    axes[-1, col].set_xlabel("Time (s)")

fig.suptitle("Features over time (max-pooled across channels)")
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="outside lower center")
plt.show()
fig.savefig("figures/features_over_time.png", dpi=150, bbox_inches="tight")


In [ ]:
"""Line length per channel as heatmaps — side by side."""
from matplotlib.colors import LogNorm

DATA_DIR = "datasets/swec-ethz"
CASES = [("03", "84h"), ("18", "93h")]
norm = LogNorm(vmin=50, vmax=2e5)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
labelled = False
im = None
for col, (patient, RECORDING) in enumerate(CASES):
    fs, seizure_intervals = load_info(DATA_DIR, patient)
    signal = load_recording(DATA_DIR, patient, RECORDING)
    offset = recording_start_seconds(RECORDING)
    windows, start_times = window_signal(signal, fs, start_offset_sec=offset)
    n_channels = signal.shape[0]

    # Line length per channel per window: (n_windows, n_channels)
    ll = np.sum(np.abs(np.diff(windows, axis=2)), axis=2)

    im = axes[col].imshow(ll.T, aspect="auto", origin="lower",
                          extent=[start_times[0], start_times[-1], 0, n_channels],
                          cmap="viridis", norm=norm)

    # Bracket any seizure that falls in the visible time range
    for sz_start, sz_end in seizure_intervals:
        if sz_end > start_times[0] and sz_start < start_times[-1]:
            axes[col].axvline(sz_start, color="crimson", linestyle="--",
                              label="Seizure" if not labelled else None)
            axes[col].axvline(sz_end, color="crimson", linestyle="--")
            labelled = True

    axes[col].set_title(f"Patient {patient}, {RECORDING}")
    axes[col].set_xlabel("Time (s)")

axes[0].set_ylabel("Channel")
fig.colorbar(im, ax=axes, label="Line length", shrink=0.8)
fig.suptitle("Line length per channel")
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center")
plt.show()
fig.savefig("figures/line_length_heatmaps.png", dpi=150, bbox_inches="tight")